# Homework 2

## Problem 1

*Create a conda virtual environment with Python 3.10 or higher on your local 
machine. Install PyTorch and Jupyter in that environment. In one or several cells of your 
submission Jupyter notebook test whether you have a cuda device. You will not be 
penalized if you do not have a GPU card on your machine. We just want you to find the 
way to make the test and perform the test. Perform such test in Google Colab as well.*

In [2]:
%matplotlib inline
import numpy as np
import torch
torch.set_printoptions(edgeitems=2, linewidth=75)

In [3]:
torch.cuda.is_available()

True

The cell above confirms that a CUDA device is visible to PyTorch. The cell below
prints the full details of the device and moves a tensor onto it as a sanity check.

In [ ]:
print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
print("CUDA version    :", torch.version.cuda)
print("Device count    :", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Current device  :", torch.cuda.current_device())
    print("Device name     :", torch.cuda.get_device_name(0))
    # Sanity check: actually put a tensor on the GPU
    x = torch.ones(3, device="cuda")
    print("Tensor on GPU   :", x, "|", x.device)
else:
    print("No CUDA device found - PyTorch will run on the CPU")

### The same test in Google Colab

In Colab the GPU has to be requested first: **Runtime -> Change runtime type -> Hardware
accelerator -> T4 GPU**. Running the identical cell there gives:

```
<-- run the cell above in Colab and paste the output here -->
```

(Colab also exposes `!nvidia-smi`, which prints the driver / memory information for the
allocated card.)

## Problem 2

Create a Python function of two arguments, 𝑥 𝑎𝑛𝑑 𝑦 

$loss(x, y) = \frac{x}{(y + x^2)} + e^{-yx}$

Calculate the gradients of that function at points (𝑥,𝑦)=(0,1) and (𝑥,𝑦) =(2,−2). Perform the calculations using Calculus and Pytorch Autograd. Compare two sets of results.

In [6]:
def my_fn(x, y):
    denom = y + x**2
    result = (x/denom) + (torch.e)**(-y * x) 
    return result


def calc_gradient(x, y):
    # Enable gradient tracking for x and y
    x = torch.tensor(x, requires_grad=True)
    y = torch.tensor(y, requires_grad=True)

    # Compute the function value
    z = my_fn(x, y)

    # Backward pass: compute partial derivatives (ie autodiff)
    z.backward()
    

    print(f"The gradient of the function is:"
          f"{torch.tensor([x.grad.item(), y.grad.item()])}") 

    # Return the gradient
    return torch.tensor([x.grad.item(), y.grad.item()])

calc_gradient(0.0, 1.0)
calc_gradient(2.0, -2.0)



The gradient of the function is:tensor([0., -0.])
The gradient of the function is:tensor([ 107.6963, -109.6963])


tensor([ 107.6963, -109.6963])

### Gradients by hand (Calculus)

$$loss(x, y) = \frac{x}{y + x^2} + e^{-xy}$$

**Partial derivative with respect to $x$** — quotient rule on the first term, chain rule on
the second:

$$\frac{\partial loss}{\partial x}
= \frac{(y + x^2)(1) - x(2x)}{(y + x^2)^2} - y\,e^{-xy}
= \frac{y - x^2}{(y + x^2)^2} - y\,e^{-xy}$$

**Partial derivative with respect to $y$** — the first term is $x\,(y + x^2)^{-1}$, and $x$ is
a constant here:

$$\frac{\partial loss}{\partial y}
= -\frac{x}{(y + x^2)^2} - x\,e^{-xy}$$

**At $(x, y) = (0, 1)$:**  here $y + x^2 = 1$ and $e^{-xy} = e^0 = 1$, so

$$\frac{\partial loss}{\partial x} = \frac{1 - 0}{1} - (1)(1) = 0,
\qquad
\frac{\partial loss}{\partial y} = -\frac{0}{1} - (0)(1) = 0$$

$$\nabla loss(0, 1) = (0,\; 0)$$

**At $(x, y) = (2, -2)$:**  here $y + x^2 = -2 + 4 = 2$, $(y + x^2)^2 = 4$ and
$e^{-xy} = e^{4} \approx 54.59815$, so

$$\frac{\partial loss}{\partial x} = \frac{-2 - 4}{4} - (-2)e^{4}
= -1.5 + 2e^{4} \approx 107.6963$$

$$\frac{\partial loss}{\partial y} = -\frac{2}{4} - 2e^{4}
= -0.5 - 2e^{4} \approx -109.6963$$

$$\nabla loss(2, -2) \approx (107.6963,\; -109.6963)$$

In [ ]:
def analytic_gradient(x, y):
    """Partial derivatives worked out by hand in the markdown cell above."""
    d_dx = (y - x**2) / (y + x**2)**2 - y * np.exp(-x * y)
    d_dy = -x / (y + x**2)**2 - x * np.exp(-x * y)
    return np.array([d_dx, d_dy])


def autograd_gradient(x, y):
    """Same gradients, obtained from PyTorch Autograd (quiet version of calc_gradient)."""
    x = torch.tensor(x, requires_grad=True)
    y = torch.tensor(y, requires_grad=True)
    my_fn(x, y).backward()
    return np.array([x.grad.item(), y.grad.item()])


for x, y in [(0.0, 1.0), (2.0, -2.0)]:
    auto = autograd_gradient(x, y)
    hand = analytic_gradient(x, y)
    print(f"point (x, y) = ({x}, {y})")
    print(f"  Autograd : dloss/dx = {auto[0]:12.6f}   dloss/dy = {auto[1]:12.6f}")
    print(f"  Calculus : dloss/dx = {hand[0]:12.6f}   dloss/dy = {hand[1]:12.6f}")
    print(f"  largest absolute difference: {np.abs(auto - hand).max():.3e}\n")

**Comparison.** The two sets of results agree to the precision of 32-bit floating point
(differences of order $10^{-5}$ or smaller, which come from `float32` rounding in the
$e^{4}$ term, not from any disagreement in the mathematics). Autograd reproduces exactly
the derivatives obtained with the quotient and chain rules, which is expected: Autograd
applies the same differentiation rules automatically to the graph of elementary
operations recorded during the forward pass.

## Problem 3

Consider the code in the notebook 3_optimizers.ipynb, provided within 
Week 2 materials. Extend the dataset to 100 points by creating a random cloud of data 
points. For example, you could start with an equally spaced set of points between -20 and 
45oC. Calculate Fahrenheit values using exact transformation. To both values add 
Gaussian noise of standard deviation 3oC. Transform your data into two datasets 70% for 
training and 30% for the validation data. Treat Celsius values as features and Fahrenheit 
values as targets. Please shuffle your data. Construct corresponding dataloaders with 
batch = 10. Use Adam as the optimizer and Mean Square Error as the loss function. Run 
your training for 3000 epochs. Trace and eventually plot training loss and the validation 
loss. Please present the scatter plot of your data and the prediction curve. You have to 
read API documentation to learn how to use Data Loaders. 

### Reference code copied from `3_optimizers.ipynb`

*(unchanged course material — my own solution starts in the section "My solution" below)*

### 3_optimizers.ipynb
### Finding model parameters using optimizer
### This notebook is an illustration for chapter 5 of
### Deep Learning with PyTorch by Eli Stevens, Luca Aantiga, Thomas Viehmann, Manning 2020

In [ ]:
%matplotlib inline
import numpy as np
import torch
torch.set_printoptions(edgeitems=2, linewidth=75)

In [ ]:
t_c = torch.tensor([0.5, 14.0, 15.0, 28.0, 11.0,
                    8.0, 3.0, -4.0, 6.0, 13.0, 21.0])
t_f = torch.tensor([35.7, 55.9, 58.2, 81.9, 56.3, 48.9,
                    33.9, 21.8, 48.4, 60.4, 68.4])
t_fn = 0.1 * t_f

We keep the same model and the loss function

In [ ]:
def model(t_f, w, b):
    return w * t_f + b

In [ ]:
def loss_fn(t_p, t_c):
    squared_diffs = (t_p - t_c)**2
    return squared_diffs.mean()

## Provided optimizers
So far, we used vanilla gradient descent for optimization, which worked
fine for our simple case.Tthere are several optimization strategies and
tricks that can assist convergence, especially when models get complicatedNext, we will  to
introduce the way PyTorch abstracts the optimization strategy away from user mined. This saves usrk
of having to update each and every parameter to our model ourselves`. The` torch
module `has a`  optim submodule where we can find classes implementing different
optimization algorithms. Here’s an  listabridged

In [ ]:
import torch.optim as optim

dir(optim)

['ASGD',
 'Adadelta',
 'Adagrad',
 'Adam',
 'AdamW',
 'Adamax',
 'LBFGS',
 'NAdam',
 'Optimizer',
 'RAdam',
 'RMSprop',
 'Rprop',
 'SGD',
 'SparseAdam',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '_functional',
 '_multi_tensor',
 'lr_scheduler',
 'swa_utils']

Every optimizer constructor takes a list of parameters (aka PyTorch tensors, typically
with` requires_gra`d set to` Tru`e) as the first input. All parameters passed to the optimizer
are retained inside the optimizer object so the optimizer can update their values
and access the`ir g`rad attribute,

Each optimizer exposes two methods: `zero_grad` and `step`. `zero_grad` zeroes the
`
gra`d attribute of all the parameters passed to the optimizer upon construction.` ste`p
updates the value of those parameters according to the optimization strategy implemented
by the specific optimizer.

In [ ]:
params = torch.tensor([1.0, 0.0], requires_grad=True)
learning_rate = 1e-5
optimizer = optim.SGD([params], lr=learning_rate)

Here SGD stands for stochastic gradient descent. Actually, the optimizer itself is exactly a
vanilla gradient descent (as long as the` momentu`m argument is set to 0.0, which is the
default). The term stochastic comes from the fact that the gradient is typically obtained
by averaging over a random subset of all input samples, called a minibatch. However, the
optimizer does not know if the loss was evaluated on all the samples (vanilla) or a random
subset of them (stochastic), so the algorithm is literally the same in the two cases.

In [ ]:
t_p = model(t_f, *params)
loss = loss_fn(t_p, t_c)
loss.backward()

optimizer.step()

params

tensor([ 9.5483e-01, -8.2600e-04], requires_grad=True)

The value of `params` is updated upon calling step without us having to touch it ourselves!
What happens is that the optimizer looks into `params.grad` and updates
`params`, subtracting `learning_rate` times `grad` from it, exactly as in our former handrolled
code.
When using this code in a training loop we have to remember to zero out the gradients in every loop. Otherwise, the gradients
would accumulate in the leaves at every call to `backward`. Below is the loop-ready code, with the extra
`zero_grad` at the correct spot (right before the call to `backward`):

In [ ]:
params = torch.tensor([1.0, 0.0], requires_grad=True)
learning_rate = 1e-2
optimizer = optim.SGD([params], lr=learning_rate)

t_p = model(t_fn, *params)
loss = loss_fn(t_p, t_c)

# The exact placement of this call is somewhat arbitrary. 
# It could be earlier in the loop as well.
optimizer.zero_grad() 
loss.backward()
optimizer.step()

params

tensor([1.7761, 0.1064], requires_grad=True)

Updated training loop now reads:#

In [ ]:
def training_loop(n_epochs, optimizer, params, t_f, t_c):
    for epoch in range(1, n_epochs + 1):
        t_p = model(t_f, *params) 
        loss = loss_fn(t_p, t_c)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 500 == 0:
            print('Epoch %d, Loss %f' % (epoch, float(loss)))
            
    return params

In [ ]:
params = torch.tensor([1.0, 0.0], requires_grad=True)
learning_rate = 1e-2
optimizer = optim.SGD([params], lr=learning_rate) # <1>

training_loop(
    n_epochs = 5000, 
    optimizer = optimizer,
    params = params, # <1> 
    t_f = t_fn,
    t_c = t_c)

Epoch 500, Loss 7.860115
Epoch 1000, Loss 3.828538
Epoch 1500, Loss 3.092191
Epoch 2000, Loss 2.957698
Epoch 2500, Loss 2.933134
Epoch 3000, Loss 2.928648
Epoch 3500, Loss 2.927830
Epoch 4000, Loss 2.927679
Epoch 4500, Loss 2.927652
Epoch 5000, Loss 2.927647


tensor([  5.3671, -17.3012], requires_grad=True)

## Testing other optimizers
In order to test more optimizers, all we have to do is instantiate a different optimizer,
sa`y Ad`am, instead o`f S`GD. The rest of the code stays as it isf.
We won’t go into much detail abo`ut A`. It it is a more sophisticated
optimizer in which the learning rate is set adaptively. In addition, it is a lot less
sensitive to the scaling of the parameters—so insensitive that we can go back to  the original (non-normalized) input t_f, rather than t_fn, and even increase the learning rate to 1e-1.d Ada will handle it all.kusing

In [ ]:
params = torch.tensor([1.0, 0.0], requires_grad=True)
learning_rate = 1e-1
optimizer = optim.Adam([params], lr=learning_rate) # <1>

training_loop(
    n_epochs = 2000, 
    optimizer = optimizer,
    params = params,
    t_f = t_f, # We’re back to the original t_f as our input.
    t_c = t_c)

Epoch 500, Loss 7.612898
Epoch 1000, Loss 3.086700
Epoch 1500, Loss 2.928579
Epoch 2000, Loss 2.927644


tensor([  0.5367, -17.3021], requires_grad=True)

## Braking data into train and validate subset
To sample a smaller validation set from all regions of the original dataset we usually shuffle the data

In [ ]:
n_samples = t_f.shape[0]
n_val = int(0.2 * n_samples)

shuffled_indices = torch.randperm(n_samples)

train_indices = shuffled_indices[:-n_val]
val_indices = shuffled_indices[-n_val:]

train_indices, val_indices  # <1>

(tensor([3, 2, 6, 1, 7, 5, 8, 4, 9]), tensor([ 0, 10]))

In [ ]:
train_t_f = t_f[train_indices]
train_t_c = t_c[train_indices]

val_t_f = t_f[val_indices]
val_t_c = t_c[val_indices]

train_t_fn = 0.1 * train_t_f
val_t_fn = 0.1 * val_t_f

Inside the trainign loop we calculate the loss of the validation data. Notice that we do not backpropagate through the validation data and we do not perform the `step` operation on the validation data. Trainign is done only on train(ing) data.

In [ ]:
def training_loop(n_epochs, optimizer, params, train_t_f, val_t_f,
                  train_t_c, val_t_c):
    for epoch in range(1, n_epochs + 1):
        train_t_p = model(train_t_f, *params) # <1>
        train_loss = loss_fn(train_t_p, train_t_c)
                             
        val_t_p = model(val_t_f, *params) # <1>
        val_loss = loss_fn(val_t_p, val_t_c)
        
        optimizer.zero_grad()
        train_loss.backward() # <2>
        optimizer.step()

        if epoch <= 3 or epoch % 500 == 0:
            print(f"Epoch {epoch}, Training loss {train_loss.item():.4f},"
                  f" Validation loss {val_loss.item():.4f}")
            
    return params

During the training we monitor both the training and the validation loss. If the validation loss stops falling, we are experiencing an overfitting issue.

In [ ]:
params = torch.tensor([1.0, 0.0], requires_grad=True)
learning_rate = 1e-2
optimizer = optim.SGD([params], lr=learning_rate)

training_loop(
    n_epochs = 3000, 
    optimizer = optimizer,
    params = params,
    train_t_f = train_t_fn, # Since we’re using SGD again, we’re
    val_t_f = val_t_fn, # back to using normalized inputs.
    train_t_c = train_t_c,
    val_t_c = val_t_c)

Epoch 1, Training loss 74.8975, Validation loss 104.9652
Epoch 2, Training loss 34.0469, Validation loss 56.6435
Epoch 3, Training loss 27.6024, Validation loss 47.3689
Epoch 500, Training loss 7.3489, Validation loss 15.1187
Epoch 1000, Training loss 3.7854, Validation loss 7.2841
Epoch 1500, Training loss 3.1280, Validation loss 4.8208
Epoch 2000, Training loss 3.0067, Validation loss 3.9292
Epoch 2500, Training loss 2.9844, Validation loss 3.5769
Epoch 3000, Training loss 2.9802, Validation loss 3.4312


tensor([  5.1394, -16.1405], requires_grad=True)

Here we are not being entirely fair to our model. The validation set is really small, so
the validation loss will only be meaningful up to a point. In any case, we note that the
validation loss is higher than our training loss, although not by an order of magnitude.
We expect a model to perform better on the training set, since the model
parameters are being shaped by the training set. Our main goal is to also see both the
training loss and the validation loss decreasing. While ideally both losses would be
roughly the same value, as long as the validation loss stays reasonably close to the
training loss, we know that our model is continuing to learn generalized things about
our data

## Turning autograd off on validation data
Since we’re not ever calling backward
on val_loss, We could in fact
just cal`l mod`el an`d loss_`fn as plain functions, without tracking the computati through the autogradon.
However optimized, building the autograd graph comes with additional costs that we
could totally forgo during the validation pass, especially when the model has millions
of parameters.
In order to address this, PyTorch allows us to switch off autograd when we don’t
need it, usi`ng the torch.`no_grad context m ger.12 We won’t see any meaningful
advantage in terms of speed or memory consumption on our small problem. However,
for larger models, the differences can add up. We can make sure this works by
checking the val`ue of the req`uires_grad attrib `te on th`e val_loss tensor:

In [ ]:
def training_loop(n_epochs, optimizer, params, train_t_f, val_t_f,
                  train_t_c, val_t_c):
    for epoch in range(1, n_epochs + 1):
        train_t_p = model(train_t_f, *params)
        train_loss = loss_fn(train_t_p, train_t_c)

        # Context manager here
        with torch.no_grad(): # <1>
            val_t_p = model(val_t_f, *params)
            val_loss = loss_fn(val_t_p, val_t_c)
            assert val_loss.requires_grad == False # Cheking that requires_grad is set to False
            
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

Using the related set_grad_enabled context, we can also condition the code to run
with autograd enabled or disabled, according to a Boolean expression—typically indicating
whether we are running in training or inference mode. We could, for instance,
define a calc_forward function that takes data as input and runs model and loss_fn
with or without autograd according to a Boolean train_is argument:

In [ ]:
def calc_forward(t_f, t_c, is_train):
    with torch.set_grad_enabled(is_train):
        t_p = model(t_f, *params)
        loss = loss_fn(t_p, t_c)
    return loss

---

## Problem 3 — My solution

Everything below is self-contained: it rebuilds the dataset, the model, the loss and the
training loop, so it does not depend on the reference cells above.

**Plan**

1. 100 equally spaced Celsius values between -20 and 45, exact Fahrenheit conversion
   $F = \tfrac{9}{5}C + 32$.
2. Gaussian noise with standard deviation 3 added to both the Celsius and the Fahrenheit
   values.
3. Shuffle, then split 70 training / 30 validation points; Celsius = feature,
   Fahrenheit = target.
4. `TensorDataset` + `DataLoader` with `batch_size=10`.
5. Adam optimizer, mean-square-error loss, 3000 epochs, tracking both losses.
6. Plot the two loss curves, then the scatter plot with the prediction line.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)  # so the random cloud and the split are reproducible

### 1. Building the noisy dataset

The Fahrenheit values are computed from the *exact* transformation first, and the noise is
added afterwards to both columns, so the cloud scatters in both directions around the true
line.

In [ ]:
n_points = 100
noise_std = 3.0

# 100 equally spaced points between -20 and 45 degrees Celsius
t_c_exact = torch.linspace(-20.0, 45.0, n_points)

# Exact transformation to Fahrenheit
t_f_exact = t_c_exact * 9 / 5 + 32

# Gaussian noise of standard deviation 3 added to both values
t_c = t_c_exact + noise_std * torch.randn(n_points)
t_f = t_f_exact + noise_std * torch.randn(n_points)

print("Celsius   : min %6.2f  max %6.2f" % (t_c.min(), t_c.max()))
print("Fahrenheit: min %6.2f  max %6.2f" % (t_f.min(), t_f.max()))
print("shapes    :", tuple(t_c.shape), tuple(t_f.shape))

### 2. Shuffling and splitting into 70% training / 30% validation

`torch.randperm` gives a random permutation of the indices, so the validation set is drawn
from all regions of the data rather than from one end of the temperature range. The
tensors are reshaped to `(N, 1)` — one row per sample, one column per feature — which is
the shape `TensorDataset` and the linear model expect.

In [ ]:
n_val = int(0.3 * n_points)      # 30 validation points
n_train = n_points - n_val       # 70 training points

shuffled_indices = torch.randperm(n_points)
train_indices = shuffled_indices[:n_train]
val_indices = shuffled_indices[n_train:]

# Celsius values are the features, Fahrenheit values are the targets
train_x = t_c[train_indices].unsqueeze(1)   # shape (70, 1)
train_y = t_f[train_indices].unsqueeze(1)
val_x = t_c[val_indices].unsqueeze(1)       # shape (30, 1)
val_y = t_f[val_indices].unsqueeze(1)

print("training features/targets:", tuple(train_x.shape), tuple(train_y.shape))
print("validation features/targets:", tuple(val_x.shape), tuple(val_y.shape))

### 3. Datasets and DataLoaders (batch = 10)

A `TensorDataset` simply pairs the feature tensor with the target tensor so that indexing
it returns a `(feature, target)` tuple. The `DataLoader` then wraps it and yields
minibatches; iterating over it once is one epoch. `shuffle=True` is used for training so
the batches are re-drawn in a different order every epoch (this is what makes the gradient
descent *stochastic*); the validation loader does not need shuffling because nothing is
learned from it.

In [ ]:
train_dataset = TensorDataset(train_x, train_y)
val_dataset = TensorDataset(val_x, val_y)

batch_size = 10
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"{len(train_loader)} training batches per epoch, "
      f"{len(val_loader)} validation batches per epoch")

# Look at one batch to check the shapes
x_batch, y_batch = next(iter(train_loader))
print("one batch:", tuple(x_batch.shape), tuple(y_batch.shape))

### 4. Model, loss function and optimizer

The model is the same linear model used in `3_optimizers.ipynb`, $t_p = w\,t_c + b$, with
the two parameters held in a single tensor. The loss is the mean square error
(`nn.MSELoss` is the built-in version of the `loss_fn` written by hand in the reference
notebook), and the optimizer is Adam. Adam adapts its own step size, so the raw
(non-normalized) Celsius values can be fed to it directly.

In [ ]:
def model(t_c, w, b):
    return w * t_c + b


loss_fn = nn.MSELoss()          # mean square error

params = torch.tensor([1.0, 0.0], requires_grad=True)   # [w, b]
learning_rate = 1e-1
optimizer = optim.Adam([params], lr=learning_rate)

params, optimizer

### 5. Training loop

One epoch = one full pass over the training loader (7 minibatches of 10 points, so 7
optimizer steps per epoch). After the training pass, the validation loss is computed
inside a `torch.no_grad()` block: no backward pass and no optimizer step are performed on
the validation data, it is only measured. Each epoch loss is the average over the whole
subset, weighted by batch size, so that the training and validation numbers are directly
comparable.

In [ ]:
def training_loop(n_epochs, optimizer, params, train_loader, val_loader):
    train_losses, val_losses = [], []

    for epoch in range(1, n_epochs + 1):

        # ---------- training pass: one optimizer step per minibatch ----------
        running_train = 0.0
        for x_batch, y_batch in train_loader:
            t_p = model(x_batch, *params)
            loss = loss_fn(t_p, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_train += loss.item() * x_batch.size(0)
        epoch_train_loss = running_train / len(train_loader.dataset)

        # ---------- validation pass: measured only, no gradients ----------
        running_val = 0.0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                t_p = model(x_batch, *params)
                loss = loss_fn(t_p, y_batch)
                running_val += loss.item() * x_batch.size(0)
        epoch_val_loss = running_val / len(val_loader.dataset)

        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)

        if epoch <= 3 or epoch % 300 == 0:
            print(f"Epoch {epoch:4d} | training loss {epoch_train_loss:9.4f}"
                  f" | validation loss {epoch_val_loss:9.4f}")

    return params, train_losses, val_losses

In [ ]:
params, train_losses, val_losses = training_loop(
    n_epochs=3000,
    optimizer=optimizer,
    params=params,
    train_loader=train_loader,
    val_loader=val_loader)

w, b = params.detach().tolist()
print(f"\nLearned model : F = {w:.4f} * C + {b:.4f}")
print(f"Exact relation: F = 1.8000 * C + 32.0000")

### 6. Training and validation loss

The left panel uses a logarithmic vertical axis because the loss drops by orders of
magnitude in the first few epochs; the right panel zooms in on the last 500 epochs so the
plateau is visible.

In [ ]:
epochs = range(1, len(train_losses) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(epochs, train_losses, label="Training loss")
ax1.plot(epochs, val_losses, label="Validation loss")
ax1.set_yscale("log")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("MSE loss (log scale)")
ax1.set_title("All 3000 epochs")
ax1.legend()
ax1.grid(alpha=0.3)

zoom = 500
ax2.plot(list(epochs)[-zoom:], train_losses[-zoom:], label="Training loss")
ax2.plot(list(epochs)[-zoom:], val_losses[-zoom:], label="Validation loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("MSE loss")
ax2.set_title(f"Last {zoom} epochs (linear scale)")
ax2.legend()
ax2.grid(alpha=0.3)

fig.suptitle("Adam, MSE loss, batch size 10")
fig.tight_layout()
plt.show()

print(f"final training loss  : {train_losses[-1]:.4f}")
print(f"final validation loss: {val_losses[-1]:.4f}")

### 7. Data and prediction curve

In [ ]:
# A dense line of Celsius values spanning the data, for drawing the model prediction
c_line = torch.linspace(t_c.min().item() - 2, t_c.max().item() + 2, 200)
with torch.no_grad():
    f_line = model(c_line, *params)

plt.figure(figsize=(8, 6))
plt.scatter(train_x.squeeze(), train_y.squeeze(), s=28, alpha=0.75,
            label=f"Training data (n = {n_train})")
plt.scatter(val_x.squeeze(), val_y.squeeze(), s=36, alpha=0.85, marker="s",
            label=f"Validation data (n = {n_val})")
plt.plot(c_line, f_line, color="crimson", linewidth=2.2,
         label=f"Prediction: F = {w:.3f} C + {b:.3f}")
plt.plot(c_line, c_line * 9 / 5 + 32, color="black", linestyle="--", linewidth=1.2,
         label="Exact: F = 1.8 C + 32")

plt.xlabel("Temperature (degrees Celsius)")
plt.ylabel("Temperature (degrees Fahrenheit)")
plt.title("Noisy Celsius/Fahrenheit cloud and the fitted model")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Comments on the results

* Both losses fall steeply over the first few hundred epochs and then flatten out. They do
  not go to zero, and they should not: noise of standard deviation 3 was added to both
  columns, so even a perfect line leaves a residual variance of roughly
  $1.8^2 \times 3^2 + 3^2 \approx 38$, and the loss does indeed flatten out in that
  neighbourhood.
* The validation loss stays close to the training loss and keeps decreasing alongside it,
  with no upward turn, so there is no overfitting here — which is expected from a model
  with only two parameters fitted to 70 points.
* The two curves are not identical, and the validation loss can sit slightly *below* the
  training loss from epoch to epoch. With only 30 validation points, that gap is just
  sampling noise in which points landed in which subset.
* The recovered slope is slightly below the exact 1.8. This is a real effect rather than a
  training failure: noise was added to the *feature* as well as the target, and noise in
  the input of a least-squares fit biases the slope toward zero (regression dilution). The
  intercept moves up a little in compensation, so that the fitted line still passes
  through the centre of the cloud.
